In [1]:
# DAY 8 : Intervention Recommender
# GOAL  : For each student group recommend exactly what they should change firstbased on their biggest weakness

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/cleaned/student_lifestyle_engineered.csv')

In [6]:

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nStudent groups:")
print(df['student_type'].value_counts())

Loaded: 1000 rows, 28 columns

Student groups:
student_type
Struggling    478
Burned Out    286
Thriving      236
Name: count, dtype: int64


In [8]:
# the biggest weakness of each group
weakness_cols = ['stress_level', 'anxiety_score',
                 'sleep_debt', 'daily_social_media_hours',
                 'screen_time_before_sleep']

profile = df.groupby('student_type')[weakness_cols].mean().round(2)

print("=" * 55)
print("WEAKNESS PROFILE PER GROUP")
print("=" * 55)
print(profile)

WEAKNESS PROFILE PER GROUP
              stress_level  anxiety_score  sleep_debt  \
student_type                                            
Burned Out            7.52           7.65        2.77   
Struggling            5.62           5.70        1.57   
Thriving              3.76           3.54        0.62   

              daily_social_media_hours  screen_time_before_sleep  
student_type                                                      
Burned Out                        3.39                      1.73  
Struggling                        2.87                      1.39  
Thriving                          2.21                      1.14  


In [9]:
# Rule based recommendation engine

def recommend(student_type):
    recommendations = {
        'Burned Out': {
            'priority'    : 'URGENT — Immediate Action Needed',
            'biggest_problem' : 'High stress + severe sleep deprivation',
            'step_1'      : 'Sleep → Target minimum 7 hours tonight',
            'step_2'      : 'Stress → 10 min meditation before bed',
            'step_3'      : 'Social Media → Limit to 1 hour per day',
            'step_4'      : 'Screen → No screens 1 hour before sleep',
            'expected'    : 'GPA improvement in 3-4 weeks'
        },
        'Struggling': {
            'priority'    : 'MODERATE — Small Changes Needed',
            'biggest_problem' : 'Moderate stress + inconsistent sleep',
            'step_1'      : 'Sleep → Fix sleep schedule, same time daily',
            'step_2'      : 'Stress → Short walk or exercise 3x per week',
            'step_3'      : 'Social Media → Reduce by 30 min per day',
            'step_4'      : 'Study → Add 30 min focused study daily',
            'expected'    : 'GPA improvement in 2-3 weeks'
        },
        'Thriving': {
            'priority'    : 'GOOD — Maintain Your Routine',
            'biggest_problem' : 'No major problems detected',
            'step_1'      : 'Sleep → Keep maintaining 7+ hours',
            'step_2'      : 'Stress → Continue current coping methods',
            'step_3'      : 'Social Media → Current usage is healthy',
            'step_4'      : 'Study → Challenge yourself with harder goals',
            'expected'    : 'Continue thriving — you are the top 24%'
        }
    }
    return recommendations[student_type]

# Test it for all 3 groups
for group in ['Burned Out', 'Struggling', 'Thriving']:
    r = recommend(group)
    print(f"\n{'='*55}")
    print(f"STUDENT TYPE : {group}")
    print(f"{'='*55}")
    for key, value in r.items():
        print(f"{key:20} : {value}")


STUDENT TYPE : Burned Out
priority             : URGENT — Immediate Action Needed
biggest_problem      : High stress + severe sleep deprivation
step_1               : Sleep → Target minimum 7 hours tonight
step_2               : Stress → 10 min meditation before bed
step_3               : Social Media → Limit to 1 hour per day
step_4               : Screen → No screens 1 hour before sleep
expected             : GPA improvement in 3-4 weeks

STUDENT TYPE : Struggling
priority             : MODERATE — Small Changes Needed
biggest_problem      : Moderate stress + inconsistent sleep
step_1               : Sleep → Fix sleep schedule, same time daily
step_2               : Stress → Short walk or exercise 3x per week
step_3               : Social Media → Reduce by 30 min per day
step_4               : Study → Add 30 min focused study daily
expected             : GPA improvement in 2-3 weeks

STUDENT TYPE : Thriving
priority             : GOOD — Maintain Your Routine
biggest_problem      : No

In [10]:
#Given a student's values predict their typee
def predict_student(stress, anxiety, sleep, mood):
    
    # Calculate engineered features
    sleep_debt = 8 - sleep
    stress_sleep_ratio = stress / sleep
    state_score = (stress + anxiety + (10 - mood)) / 3
    
    # Simple rule based classification
    if stress >= 7 and sleep <= 5.5:
        student_type = 'Burned Out'
    elif stress <= 4.5 and sleep >= 7:
        student_type = 'Thriving'
    else:
        student_type = 'Struggling'
    
    print(f"\n{'='*55}")
    print(f"STUDENT ANALYSIS")
    print(f"{'='*55}")
    print(f"Stress Level  : {stress}/10")
    print(f"Anxiety Score : {anxiety}/10")
    print(f"Sleep Hours   : {sleep} hrs")
    print(f"Mood Rating   : {mood}/10")
    print(f"Sleep Debt    : {sleep_debt:.1f} hrs")
    print(f"State Score   : {state_score:.2f}/10")
    print(f"\n→ You are : {student_type}")
    
    rec = recommend(student_type)
    print(f"\nRECOMMENDATION:")
    for key, value in rec.items():
        print(f"  {key:20} : {value}")

# Test with a burned out student
predict_student(stress=8, anxiety=7, sleep=5, mood=3)


STUDENT ANALYSIS
Stress Level  : 8/10
Anxiety Score : 7/10
Sleep Hours   : 5 hrs
Mood Rating   : 3/10
Sleep Debt    : 3.0 hrs
State Score   : 7.33/10

→ You are : Burned Out

RECOMMENDATION:
  priority             : URGENT — Immediate Action Needed
  biggest_problem      : High stress + severe sleep deprivation
  step_1               : Sleep → Target minimum 7 hours tonight
  step_2               : Stress → 10 min meditation before bed
  step_3               : Social Media → Limit to 1 hour per day
  step_4               : Screen → No screens 1 hour before sleep
  expected             : GPA improvement in 3-4 weeks
